# MVP-1 — Dos Etapas: Pretrain (2,249 imgs) → Finetune (393 imgs)

**Problema:** Fold 2 colapsa (ρ=0.399-0.617) porque tiene solo 86 imágenes
de 3 donantes de fases distintas al train. No hay suficiente diversidad
para aprender representaciones generales solo con Control 10x.

**Solución:** Preentrenar el embedding head con TODAS las imágenes (~2,249)
incluyendo SURF1 y todos los treatments. Luego finetunear solo el pdl_head
sobre Control 10x para métricas limpias.

| Etapa | Datos | Qué entrena | Objetivo |
|-------|-------|-------------|----------|
| 1. Pretrain | ~2,249 imgs (todo) | embed_head + pdl_head | Representaciones generales |
| 2. Finetune | ~393 imgs (Control 10x) | solo pdl_head | Métricas limpias |

---

## SECCIÓN 0 — CONFIG

In [10]:
import os

# ============================================================
# >>> AJUSTAR ESTAS RUTAS <<<
# ============================================================
MANIFEST_PRETRAIN = "/Users/JCB/Documentos/Proyecto Integrador/data/manifests/manifest_mvp1_pretrain_20260319_153838.csv"
MANIFEST_FINETUNE = "/Users/JCB/Documentos/Proyecto Integrador/data/manifests/manifest_mvp1_finetune_20260319_153838.csv"
CSV_CENTRAL       = "/Users/JCB/Documentos/Proyecto Integrador/data/Lifespan_Study_Data.csv"
IMAGE_ROOT        = "/Volumes/SanDisk SSD 1TB/Storage/Data/Cellular_Lifespan_Study_Brightfield_Images"
OUTPUT_DIR        = "/Users/JCB/Documentos/Proyecto Integrador/results/mvp1_twostage"

# ============================================================
# COLUMNAS
# ============================================================
COL_IMG_PATH      = "img_path"
COL_PDL_NORM      = "pdl_norm"
COL_PDL_RAW       = "pdl"
COL_FOLD          = "fold"
COL_CELL_LINE     = "cell_line"
COL_STUDY_PART    = "study_part"
COL_MAGNIFICATION = "magnification"
COL_SAMPLE_ID     = "sample_id"
COL_TELOMERE      = "telomere_length"
COL_MTDNA         = "mtdna_cn"

# ============================================================
# HIPERPARÁMETROS
# ============================================================
BATCH_SIZE    = 16
EMBEDDING_DIM = 256
IMG_SIZE      = 224
SEED          = 42

# Stage 1: Pretrain
EPOCHS_S1     = 25
PATIENCE_S1   = 8
LR_S1         = 1e-3
LAMBDA_PDL    = 1.0
LAMBDA_RANK   = 0.3
LAMBDA_CONS   = 0.2
LAMBDA_DANN_MAG = 0.3
DANN_MAX_MAG  = 0.5

# Stage 2: Finetune
EPOCHS_S2     = 30
PATIENCE_S2   = 7
LR_S2         = 5e-4     # más bajo — solo afinamos pdl_head

WEIGHT_DECAY  = 1e-4

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Config cargada. Output → {OUTPUT_DIR}")

✅ Config cargada. Output → /Users/JCB/Documentos/Proyecto Integrador/results/mvp1_twostage


## SECCIÓN 1 — IMPORTS

In [11]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import json
from collections import defaultdict
import copy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Function
import torchvision.transforms as T
import torchvision.models as models

from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA

from PIL import Image

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("🖥  Device: Apple MPS")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"🖥  Device: CUDA — {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("🖥  Device: CPU")

🖥  Device: Apple MPS


## SECCIÓN 2 — CARGAR AMBOS MANIFESTS

In [12]:
def load_manifest(path, filter_10x_only=False, exclude_4x=True):
    """Carga y prepara un manifest."""
    df = pd.read_csv(path)
    print(f"  Cargado: {df.shape[0]} filas")

    # Resolver rutas
    if not os.path.isabs(df[COL_IMG_PATH].iloc[0]):
        df["_img_abs"] = df[COL_IMG_PATH].apply(lambda p: os.path.join(IMAGE_ROOT, p))
    else:
        df["_img_abs"] = df[COL_IMG_PATH]

    # Verificar existencia
    exists_mask = df["_img_abs"].apply(os.path.exists)
    n_miss = (~exists_mask).sum()
    if n_miss > 0:
        print(f"  ⚠️ {n_miss} imágenes no encontradas — filtrando")
    df = df[exists_mask].reset_index(drop=True)

    # Filtros de magnificación
    if exclude_4x:
        df = df[df[COL_MAGNIFICATION].astype(str).isin(["10", "20"])].reset_index(drop=True)
    if filter_10x_only:
        df = df[df[COL_MAGNIFICATION].astype(str) == "10"].reset_index(drop=True)

    print(f"  Final: {len(df)} imágenes")
    return df


print("📦 PRETRAIN manifest:")
df_pretrain = load_manifest(MANIFEST_PRETRAIN, filter_10x_only=False, exclude_4x=True)

print("\n📦 FINETUNE manifest (Control, sin SURF1, solo 10x):")
df_finetune = load_manifest(MANIFEST_FINETUNE, filter_10x_only=True, exclude_4x=True)

# Codificar variables globales (usando pretrain como referencia)
all_study_parts = sorted(df_pretrain[COL_STUDY_PART].unique())
sp_to_idx = {sp: i for i, sp in enumerate(all_study_parts)}
N_STUDY_PARTS = len(all_study_parts)

all_mags = sorted(df_pretrain[COL_MAGNIFICATION].astype(str).unique())
mag_to_idx = {m: i for i, m in enumerate(all_mags)}
N_MAGS = len(all_mags)

all_cell_lines = sorted(df_pretrain[COL_CELL_LINE].unique())
cl_to_idx = {cl: i for i, cl in enumerate(all_cell_lines)}
N_CELL_LINES = len(all_cell_lines)

# Aplicar codificación a ambos manifests
for df_tmp in [df_pretrain, df_finetune]:
    df_tmp["_sp_idx"] = df_tmp[COL_STUDY_PART].map(sp_to_idx)
    df_tmp["_mag_idx"] = df_tmp[COL_MAGNIFICATION].astype(str).map(mag_to_idx)
    df_tmp["_cl_idx"] = df_tmp[COL_CELL_LINE].map(cl_to_idx)

print(f"\n📊 Pretrain: {len(df_pretrain)} imgs, {len(df_pretrain[COL_CELL_LINE].unique())} cell lines")
print(f"📊 Finetune: {len(df_finetune)} imgs, {len(df_finetune[COL_CELL_LINE].unique())} cell lines")
print(f"   Study parts: {all_study_parts}")
print(f"   Magnificaciones: {all_mags}")

# Fold stats para ambos
print(f"\n📊 Folds PRETRAIN:")
for fold in sorted(df_pretrain[COL_FOLD].unique()):
    sub = df_pretrain[df_pretrain[COL_FOLD] == fold]
    print(f"   Fold {fold}: {len(sub)} imgs | {sorted(sub[COL_CELL_LINE].unique())}")

print(f"\n📊 Folds FINETUNE (10x):")
for fold in sorted(df_finetune[COL_FOLD].unique()):
    sub = df_finetune[df_finetune[COL_FOLD] == fold]
    print(f"   Fold {fold}: {len(sub)} imgs | {sorted(sub[COL_CELL_LINE].unique())}")

📦 PRETRAIN manifest:
  Cargado: 2249 filas
  Final: 2247 imágenes

📦 FINETUNE manifest (Control, sin SURF1, solo 10x):
  Cargado: 763 filas
  Final: 393 imágenes

📊 Pretrain: 2247 imgs, 9 cell lines
📊 Finetune: 393 imgs, 6 cell lines
   Study parts: [1, 2, 3, 5]
   Magnificaciones: ['10', '20']

📊 Folds PRETRAIN:
   Fold 0.0: 485 imgs | ['hFB12']
   Fold 1.0: 1159 imgs | ['hFB13', 'hFB14', 'hFB6', 'hFB8']
   Fold 2.0: 603 imgs | ['hFB1', 'hFB11', 'hFB2', 'hFB7']

📊 Folds FINETUNE (10x):
   Fold 0.0: 127 imgs | ['hFB12']
   Fold 1.0: 217 imgs | ['hFB13', 'hFB14']
   Fold 2.0: 49 imgs | ['hFB1', 'hFB11', 'hFB2']


/var/folders/fy/k8jygkfn6hd63_wrlzhk82c40000gp/T/ipykernel_68411/1959026463.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_img_abs"] = df[COL_IMG_PATH]
/var/folders/fy/k8jygkfn6hd63_wrlzhk82c40000gp/T/ipykernel_68411/1959026463.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_img_abs"] = df[COL_IMG_PATH]
/var/folders/fy/k8jygkfn6hd63_wrlzhk82c40000gp/T/ipykernel_68411/1959026463.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h

## SECCIÓN 3 — DATASET Y TRANSFORMS

In [13]:
class TwoStageDataset(Dataset):
    def __init__(self, dataframe, transform, transform_aug=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.transform_aug = transform_aug or transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["_img_abs"])
        if img.mode != "RGB":
            img = img.convert("RGB")

        img1 = self.transform(img)
        img2 = self.transform_aug(img)

        pdl = torch.tensor(row[COL_PDL_NORM], dtype=torch.float32)
        cl_idx = torch.tensor(row["_cl_idx"], dtype=torch.long)
        sp_idx = torch.tensor(row["_sp_idx"], dtype=torch.long)
        mag_idx = torch.tensor(row["_mag_idx"], dtype=torch.long)

        return img1, img2, pdl, cl_idx, sp_idx, mag_idx


train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_transform_aug = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("✅ Dataset y transforms definidos")

✅ Dataset y transforms definidos


## SECCIÓN 4 — MODELO

In [16]:
class GradientReversalFn(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


class PDLEncoderTwoStage(nn.Module):
    """
    ResNet-34 (frozen) + embed_head + pdl_head + dann_mag.
    - Stage 1: entrena embed_head + pdl_head + dann_mag
    - Stage 2: congela embed_head, entrena solo pdl_head
    """

    def __init__(self, embedding_dim=256, n_mags=2, freeze_backbone=True):
        super().__init__()

        backbone = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        backbone_out = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.embed_head = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.pdl_head = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

        self.dann_mag = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, n_mags),
        )

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"   Params entrenables: {trainable:,}")

    def forward(self, x, alpha_mag=0.0):
        features = self.backbone(x)
        embedding = self.embed_head(features)
        pdl_hat = self.pdl_head(embedding).squeeze(-1)

        z_rev = GradientReversalFn.apply(embedding, alpha_mag)
        mag_logits = self.dann_mag(z_rev)

        return pdl_hat, embedding, mag_logits

    def freeze_for_finetune(self):
        """Stage 2: congela backbone + dann_mag. 
        Deja embed_head + pdl_head entrenables con LR bajo."""
        for param in self.dann_mag.parameters():
            param.requires_grad = False
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"   🔒 dann_mag congelado. Entrenables: {trainable:,} (embed_head + pdl_head)")


print("✅ Modelo definido")

✅ Modelo definido


## SECCIÓN 5 — PÉRDIDAS

In [15]:
def ranking_loss_scalar(pdl_hat, pdl_true, cl_idx):
    """Ranking intra cell_line usando output escalar."""
    loss = torch.tensor(0.0, device=pdl_hat.device)
    n_pairs = 0
    for cl in cl_idx.unique():
        mask = cl_idx == cl
        if mask.sum() < 2:
            continue
        scores = pdl_hat[mask]
        targets = pdl_true[mask]
        n = scores.shape[0]
        for i in range(n):
            for j in range(i + 1, n):
                diff = abs(targets[i] - targets[j])
                if diff < 0.05:
                    continue
                if targets[i] < targets[j]:
                    loss += torch.relu(diff * 0.3 - (scores[j] - scores[i]))
                else:
                    loss += torch.relu(diff * 0.3 - (scores[i] - scores[j]))
                n_pairs += 1
    return loss / max(n_pairs, 1)


def consistency_loss(emb1, emb2):
    return torch.mean((emb1 - emb2) ** 2)


def dann_alpha_schedule(epoch, max_epochs, max_alpha):
    p = epoch / max_epochs
    return max_alpha * (2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0)


print("✅ Pérdidas definidas")

✅ Pérdidas definidas


## SECCIÓN 6 — STAGE 1: PRETRAIN
Entrena embed_head + pdl_head + DANN(mag) sobre ~2,249 imágenes.
3-fold CV: para cada fold, guarda el mejor embed_head.

In [7]:
criterion_pdl = nn.L1Loss()
criterion_ce = nn.CrossEntropyLoss()
folds = sorted(df_pretrain[COL_FOLD].unique())

print("=" * 60)
print("  STAGE 1 — PRETRAIN (todas las imágenes)")
print("=" * 60)

# Guardamos el state_dict del embed_head por fold
pretrain_states = {}

for val_fold in folds:
    print(f"\n{'─'*55}")
    print(f"  S1 | Fold {val_fold}")
    print(f"{'─'*55}")

    df_tr = df_pretrain[df_pretrain[COL_FOLD] != val_fold]
    df_va = df_pretrain[df_pretrain[COL_FOLD] == val_fold]
    print(f"  Train: {len(df_tr)} | Val: {len(df_va)} | "
          f"Val cells: {sorted(df_va[COL_CELL_LINE].unique())}")

    ds_tr = TwoStageDataset(df_tr, train_transform, train_transform_aug)
    ds_va = TwoStageDataset(df_va, val_transform, val_transform)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = PDLEncoderTwoStage(EMBEDDING_DIM, N_MAGS).to(DEVICE)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_S1, weight_decay=WEIGHT_DECAY)

    best_mae = float("inf")
    patience_counter = 0

    for epoch in range(EPOCHS_S1):
        model.train()
        alpha_mag = dann_alpha_schedule(epoch, EPOCHS_S1, DANN_MAX_MAG)

        for batch in dl_tr:
            img1, img2, pdl, cl_idx, sp_idx, mag_idx = batch
            img1, img2 = img1.to(DEVICE), img2.to(DEVICE)
            pdl, cl_idx = pdl.to(DEVICE), cl_idx.to(DEVICE)
            mag_idx = mag_idx.to(DEVICE)

            optimizer.zero_grad()

            pdl_hat, emb1, mag_logits = model(img1, alpha_mag=alpha_mag)

            loss = LAMBDA_PDL * criterion_pdl(pdl_hat, pdl)
            loss += LAMBDA_RANK * ranking_loss_scalar(pdl_hat, pdl, cl_idx)

            _, emb2, _ = model(img2, alpha_mag=0.0)
            loss += LAMBDA_CONS * consistency_loss(emb1, emb2)

            loss += LAMBDA_DANN_MAG * criterion_ce(mag_logits, mag_idx)

            loss.backward()
            optimizer.step()

        # Eval
        model.eval()
        val_pdl, val_hat = [], []
        with torch.no_grad():
            for batch in dl_va:
                img1 = batch[0].to(DEVICE)
                pdl_hat, _, _ = model(img1, alpha_mag=0.0)
                val_pdl.extend(batch[2].numpy())
                val_hat.extend(pdl_hat.cpu().numpy())

        mae = np.mean(np.abs(np.array(val_pdl) - np.array(val_hat)))
        rho, _ = spearmanr(val_pdl, val_hat)

        improved = ""
        if mae < best_mae:
            best_mae = mae
            patience_counter = 0
            improved = "✓"
            # Guardar estado del embed_head
            pretrain_states[val_fold] = copy.deepcopy(model.embed_head.state_dict())
            torch.save(model.state_dict(),
                       os.path.join(OUTPUT_DIR, f"s1_best_fold{val_fold}.pt"))
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or improved or epoch == 0:
            print(f"  Ep {epoch+1:2d} | MAE={mae:.4f} | ρ={rho:.3f} | "
                  f"α={alpha_mag:.2f} {improved}")

        if patience_counter >= PATIENCE_S1:
            print(f"  Early stopping ep {epoch+1}")
            break

    print(f"  📊 S1 Fold {val_fold} best MAE: {best_mae:.4f}")

print(f"\n✅ Stage 1 completo. Embed heads guardados para {len(pretrain_states)} folds.")

  STAGE 1 — PRETRAIN (todas las imágenes)

───────────────────────────────────────────────────────
  S1 | Fold 0.0
───────────────────────────────────────────────────────
  Train: 1762 | Val: 485 | Val cells: ['hFB12']
   Params entrenables: 164,419
  Ep  1 | MAE=0.1772 | ρ=0.652 | α=0.00 ✓
  Ep  3 | MAE=0.1734 | ρ=0.658 | α=0.19 ✓
  Ep  4 | MAE=0.1521 | ρ=0.655 | α=0.27 ✓
  Ep  5 | MAE=0.1758 | ρ=0.645 | α=0.33 
  Ep 10 | MAE=0.1781 | ρ=0.686 | α=0.47 
  Early stopping ep 12
  📊 S1 Fold 0.0 best MAE: 0.1521

───────────────────────────────────────────────────────
  S1 | Fold 1.0
───────────────────────────────────────────────────────
  Train: 1088 | Val: 1159 | Val cells: ['hFB13', 'hFB14', 'hFB6', 'hFB8']
   Params entrenables: 164,419
  Ep  1 | MAE=0.1799 | ρ=0.512 | α=0.00 ✓
  Ep  2 | MAE=0.1789 | ρ=0.546 | α=0.10 ✓
  Ep  3 | MAE=0.1714 | ρ=0.560 | α=0.19 ✓
  Ep  5 | MAE=0.1897 | ρ=0.562 | α=0.33 
  Ep  7 | MAE=0.1712 | ρ=0.580 | α=0.42 ✓
  Ep 10 | MAE=0.1925 | ρ=0.564 | α=0.47 
  

## SECCIÓN 7 — STAGE 2: FINETUNE
Congela embed_head (de Stage 1). Entrena solo pdl_head sobre Control 10x.
Métricas oficiales aquí.

In [17]:
print("\n" + "=" * 60)
print("  STAGE 2 — FINETUNE (Control 10x)")
print("=" * 60)

folds_ft = sorted(df_finetune[COL_FOLD].unique())
results_per_fold = {}
all_embeddings = []

for val_fold in folds_ft:
    print(f"\n{'─'*55}")
    print(f"  S2 | Fold {val_fold}")
    print(f"{'─'*55}")

    df_tr = df_finetune[df_finetune[COL_FOLD] != val_fold]
    df_va = df_finetune[df_finetune[COL_FOLD] == val_fold]
    print(f"  Train: {len(df_tr)} | Val: {len(df_va)} | "
          f"Val cells: {sorted(df_va[COL_CELL_LINE].unique())}")

    ds_tr = TwoStageDataset(df_tr, train_transform, train_transform)
    ds_va = TwoStageDataset(df_va, val_transform, val_transform)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    trivial_mae = np.mean(np.abs(
        df_va[COL_PDL_NORM].values - df_tr[COL_PDL_NORM].mean()))

    # Crear modelo y cargar embed_head de Stage 1
    model = PDLEncoderTwoStage(EMBEDDING_DIM, N_MAGS).to(DEVICE)

    if val_fold in pretrain_states:
        model.embed_head.load_state_dict(pretrain_states[val_fold])
        print(f"  ✅ embed_head cargado de Stage 1 fold {val_fold}")
    else:
        print(f"  ⚠️ No hay estado S1 para fold {val_fold} — usando random")

    # Congelar embed_head + dann_mag
    model.freeze_for_finetune()

    # Solo entrenar pdl_head
   # optimizer = optim.AdamW(
   #     filter(lambda p: p.requires_grad, model.parameters()),
   #     lr=LR_S2, weight_decay=WEIGHT_DECAY)
    
    # LR diferenciado: embed_head bajo, pdl_head normal
    optimizer = optim.AdamW([
        {"params": model.embed_head.parameters(), "lr": LR_S2 * 0.1},  # 10x menor
        {"params": model.pdl_head.parameters(), "lr": LR_S2},
    ], weight_decay=WEIGHT_DECAY)


    best_mae = float("inf")
    patience_counter = 0

    for epoch in range(EPOCHS_S2):
        model.train()
        for batch in dl_tr:
            img1 = batch[0].to(DEVICE)
            pdl = batch[2].to(DEVICE)
            cl_idx = batch[3].to(DEVICE)
            optimizer.zero_grad()
            pdl_hat, _, _ = model(img1, alpha_mag=0.0)
            loss = criterion_pdl(pdl_hat, pdl)
            loss += LAMBDA_RANK * ranking_loss_scalar(pdl_hat, pdl, cl_idx)
            loss.backward()
            optimizer.step()

        # Eval
        model.eval()
        val_pdl, val_hat, val_emb = [], [], []
        with torch.no_grad():
            for batch in dl_va:
                img1 = batch[0].to(DEVICE)
                pdl_hat, emb, _ = model(img1, alpha_mag=0.0)
                val_pdl.extend(batch[2].numpy())
                val_hat.extend(pdl_hat.cpu().numpy())
                val_emb.append(emb.cpu().numpy())

        val_pdl = np.array(val_pdl)
        val_hat = np.array(val_hat)
        mae = np.mean(np.abs(val_pdl - val_hat))
        rho, p_val = spearmanr(val_pdl, val_hat)

        improved = ""
        if mae < best_mae:
            best_mae = mae
            patience_counter = 0
            improved = "✓"
            torch.save(model.state_dict(),
                       os.path.join(OUTPUT_DIR, f"s2_best_fold{val_fold}.pt"))
            best_emb = np.vstack(val_emb)
            best_hat = val_hat.copy()
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or improved or epoch == 0:
            print(f"  Ep {epoch+1:2d} | MAE={mae:.4f} | ρ={rho:.3f} {improved}")

        if patience_counter >= PATIENCE_S2:
            print(f"  Early stopping ep {epoch+1}")
            break

    # Final metrics
    ss_res = np.sum((val_pdl - best_hat) ** 2)
    ss_tot = np.sum((val_pdl - val_pdl.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    rho_final, p_final = spearmanr(val_pdl, best_hat)
    improvement = (1 - best_mae / trivial_mae) * 100

    # Save embeddings
    df_emb = df_va.copy().reset_index(drop=True)
    df_emb["y_pred"] = best_hat
    emb_df = pd.DataFrame(best_emb, columns=[f"emb_{i}" for i in range(best_emb.shape[1])])
    df_emb = pd.concat([df_emb, emb_df], axis=1)
    all_embeddings.append(df_emb)

    results_per_fold[val_fold] = {
        "n_val": len(df_va), "cell_lines_val": sorted(df_va[COL_CELL_LINE].unique()),
        "trivial_mae": trivial_mae, "model_mae": best_mae,
        "improvement_pct": improvement, "spearman": rho_final,
        "spearman_p": p_final, "r2": r2,
    }

    print(f"  📊 MAE={best_mae:.4f} (mejora {improvement:+.1f}%) | "
          f"ρ={rho_final:.3f} | R²={r2:.3f}")

df_embeddings = pd.concat(all_embeddings, ignore_index=True)


  STAGE 2 — FINETUNE (Control 10x)

───────────────────────────────────────────────────────
  S2 | Fold 0.0
───────────────────────────────────────────────────────
  Train: 266 | Val: 127 | Val cells: ['hFB12']
   Params entrenables: 164,419
  ✅ embed_head cargado de Stage 1 fold 0.0
   🔒 dann_mag congelado. Entrenables: 147,841 (embed_head + pdl_head)
  Ep  1 | MAE=0.2255 | ρ=0.694 ✓
  Ep  2 | MAE=0.1712 | ρ=0.698 ✓
  Ep  3 | MAE=0.1627 | ρ=0.703 ✓
  Ep  5 | MAE=0.1840 | ρ=0.705 
  Ep 10 | MAE=0.1726 | ρ=0.717 
  Early stopping ep 10
  📊 MAE=0.1627 (mejora +29.9%) | ρ=0.703 | R²=0.421

───────────────────────────────────────────────────────
  S2 | Fold 1.0
───────────────────────────────────────────────────────
  Train: 176 | Val: 217 | Val cells: ['hFB13', 'hFB14']
   Params entrenables: 164,419
  ✅ embed_head cargado de Stage 1 fold 1.0
   🔒 dann_mag congelado. Entrenables: 147,841 (embed_head + pdl_head)
  Ep  1 | MAE=0.3181 | ρ=0.598 ✓
  Ep  2 | MAE=0.3072 | ρ=0.659 ✓
  Ep  3 | M

## SECCIÓN 8 — RESULTADOS + BATCH PROBE + FUSION-READINESS

In [18]:
maes = [r["model_mae"] for r in results_per_fold.values()]
spears = [r["spearman"] for r in results_per_fold.values()]
imps = [r["improvement_pct"] for r in results_per_fold.values()]

print("=" * 70)
print("  RESULTADOS — TWO-STAGE")
print("=" * 70)

for fold, r in results_per_fold.items():
    print(f"  Fold {fold}: MAE={r['model_mae']:.4f} | ρ={r['spearman']:.3f} | "
          f"R²={r['r2']:.3f} | mejora={r['improvement_pct']:+.1f}% | "
          f"cells={', '.join(r['cell_lines_val'])}")

print(f"\n  Promedio: MAE={np.mean(maes):.4f}±{np.std(maes):.4f} | "
      f"ρ={np.mean(spears):.3f}±{np.std(spears):.3f} | mejora={np.mean(imps):+.1f}%")
print(f"  Worst fold ρ: {min(spears):.3f}")

s_ok = "✓" if np.mean(spears) >= 0.6 else "✗"
m_ok = "✓" if np.mean(imps) >= 25 else "✗"
w_ok = "✓" if min(spears) >= 0.6 else "✗"
print(f"\n  📋 PIDA: ρ≥0.6 {s_ok} | mejora≥25% {m_ok} | worst≥0.6 {w_ok}")

# --- BATCH PROBE INCREMENTAL ---
print(f"\n{'─'*60}")
print("  BATCH PROBE INCREMENTAL")
print(f"{'─'*60}")

emb_cols = [c for c in df_embeddings.columns if c.startswith("emb_")]
X_emb = df_embeddings[emb_cols].values

# Study_part
y_sp = df_embeddings[COL_STUDY_PART].values
le_sp = LabelEncoder()
y_sp_enc = le_sp.fit_transform(y_sp)

X_struct = pd.get_dummies(df_embeddings[[COL_PDL_NORM, COL_CELL_LINE]],
                          columns=[COL_CELL_LINE]).values.astype(float)

clf_a = LogisticRegression(max_iter=2000, random_state=SEED, C=1.0)
clf_a.fit(X_struct, y_sp_enc)
auc_struct = roc_auc_score(y_sp_enc, clf_a.predict_proba(X_struct),
                           multi_class="ovr", average="macro")

clf_b = LogisticRegression(max_iter=2000, random_state=SEED, C=1.0)
clf_b.fit(np.hstack([X_struct, X_emb]), y_sp_enc)
auc_full = roc_auc_score(y_sp_enc, clf_b.predict_proba(np.hstack([X_struct, X_emb])),
                         multi_class="ovr", average="macro")

clf_c = LogisticRegression(max_iter=2000, random_state=SEED, C=1.0)
clf_c.fit(X_emb, y_sp_enc)
auc_raw = roc_auc_score(y_sp_enc, clf_c.predict_proba(X_emb),
                        multi_class="ovr", average="macro")

delta_sp = auc_full - auc_struct
s1 = "✅" if delta_sp <= 0.10 else "⚠️"
print(f"  Study_part:  raw={auc_raw:.3f} | struct={auc_struct:.3f} | +z={auc_full:.3f} | Δ={delta_sp:+.3f} {s1}")

# PDL bins
pdl_bins = pd.cut(df_embeddings[COL_PDL_NORM], bins=3, labels=[0, 1, 2])
clf_pdl = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
clf_pdl.fit(X_emb, pdl_bins)
auc_pdl = roc_auc_score(pdl_bins, clf_pdl.predict_proba(X_emb),
                        multi_class="ovr", average="macro")
s2 = "✅" if auc_pdl >= 0.85 else "⚠️"
print(f"  PDL bins:    AUC={auc_pdl:.3f} {s2}")

# --- FUSION-READINESS ---
print(f"\n{'─'*60}")
print("  FUSION-READINESS")
print(f"{'─'*60}")

def partial_correlation(x, y, covariates):
    reg_x = LinearRegression().fit(covariates, x)
    res_x = x - reg_x.predict(covariates)
    reg_y = LinearRegression().fit(covariates, y)
    res_y = y - reg_y.predict(covariates)
    return spearmanr(res_x, res_y)

pca = PCA(n_components=1, random_state=SEED)
df_embeddings["emb_pc1"] = pca.fit_transform(X_emb)[:, 0]

bio_cols = [COL_TELOMERE, COL_MTDNA]
agg_dict = {"emb_pc1": "mean", COL_PDL_NORM: "first", COL_CELL_LINE: "first"}
for col in bio_cols:
    if col in df_embeddings.columns:
        agg_dict[col] = "first"

sub = df_embeddings.dropna(subset=[c for c in bio_cols if c in df_embeddings.columns])
fusion_results = {}

if len(sub) > 0:
    agg = sub.groupby(COL_SAMPLE_ID).agg(agg_dict).reset_index()
    cov = pd.get_dummies(agg[[COL_PDL_NORM, COL_CELL_LINE]],
                         columns=[COL_CELL_LINE]).values.astype(float)

    for col in bio_cols:
        if col not in agg.columns:
            continue
        mask = agg[col].notna()
        n_valid = mask.sum()
        if n_valid < 15:
            continue
        x = agg.loc[mask, "emb_pc1"].values
        y = agg.loc[mask, col].values
        c = cov[mask.values]

        rho_s, p_s = spearmanr(x, y)
        rho_p, p_p = partial_correlation(x, y, c)
        sig = "*" if p_p < 0.05 else "ns"
        print(f"  {col:25s} | simple ρ={rho_s:+.3f} | parcial ρ={rho_p:+.3f} ({sig}, p={p_p:.3e}) | n={n_valid}")
        fusion_results[col] = {"simple_rho": float(rho_s), "partial_rho": float(rho_p),
                               "partial_p": float(p_p), "n": int(n_valid)}

    rho_pdl, p_pdl = spearmanr(agg["emb_pc1"], agg[COL_PDL_NORM])
    print(f"  {'PDL (sanity)':25s} | ρ={rho_pdl:+.3f} (p={p_pdl:.2e})")

# --- COMPARACIÓN CON ANTERIORES ---
print(f"\n{'─'*60}")
print("  COMPARACIÓN")
print(f"{'─'*60}")

prev = {
    "A0_baseline_10x":  {"rho": 0.738, "worst": 0.616, "delta": None,  "mtdna": "ns"},
    "A2_rank_cons_10x": {"rho": 0.736, "worst": 0.617, "delta": 0.132, "mtdna": "0.252 *"},
    "A4_dann_cond":     {"rho": 0.638, "worst": 0.399, "delta": 0.076, "mtdna": "0.185 ns"},
}

print(f"  {'Model':22s} | {'ρ mean':>7s} | {'worst':>6s} | {'Δsp':>6s} | mtDNA parcial")
print(f"  {'-'*22}-+-{'-'*7}-+-{'-'*6}-+-{'-'*6}-+-{'-'*15}")
for name, p in prev.items():
    d = f"{p['delta']:+.3f}" if p['delta'] is not None else "  n/a"
    print(f"  {name:22s} | {p['rho']:7.3f} | {p['worst']:6.3f} | {d} | {p['mtdna']}")

mtdna_str = "n/a"
if COL_MTDNA in fusion_results:
    fr = fusion_results[COL_MTDNA]
    sig = "*" if fr["partial_p"] < 0.05 else "ns"
    mtdna_str = f"{fr['partial_rho']:+.3f} {sig}"

d_str = f"{delta_sp:+.3f}"
print(f"  {'>>> Two-Stage':22s} | {np.mean(spears):7.3f} | {min(spears):6.3f} | "
      f"{d_str} | {mtdna_str}")

# Guardar reporte
report = {
    "timestamp": datetime.now().isoformat(),
    "model": "Two-Stage: S1 pretrain(2249) + S2 finetune(Control 10x)",
    "results_per_fold": {str(k): v for k, v in results_per_fold.items()},
    "aggregated": {"mean_spearman": float(np.mean(spears)),
                   "worst_spearman": float(min(spears)),
                   "mean_mae": float(np.mean(maes)),
                   "mean_improvement": float(np.mean(imps))},
    "batch_probe": {"study_part_delta": delta_sp, "study_part_raw": auc_raw,
                    "pdl_bins": auc_pdl},
    "fusion_readiness": fusion_results,
}
with open(os.path.join(OUTPUT_DIR, "twostage_report.json"), "w") as f:
    json.dump(report, f, indent=2, default=str)
print(f"\n💾 Reporte: twostage_report.json")

# Guardar embeddings
df_embeddings.to_csv(os.path.join(OUTPUT_DIR, "embeddings_twostage.csv"), index=False)
print(f"💾 Embeddings: embeddings_twostage.csv")

  RESULTADOS — TWO-STAGE
  Fold 0.0: MAE=0.1627 | ρ=0.703 | R²=0.421 | mejora=+29.9% | cells=hFB12
  Fold 1.0: MAE=0.1584 | ρ=0.685 | R²=0.463 | mejora=+40.2% | cells=hFB13, hFB14
  Fold 2.0: MAE=0.1908 | ρ=0.559 | R²=0.316 | mejora=+26.5% | cells=hFB1, hFB11, hFB2

  Promedio: MAE=0.1706±0.0143 | ρ=0.649±0.064 | mejora=+32.2%
  Worst fold ρ: 0.559

  📋 PIDA: ρ≥0.6 ✓ | mejora≥25% ✓ | worst≥0.6 ✗

────────────────────────────────────────────────────────────
  BATCH PROBE INCREMENTAL
────────────────────────────────────────────────────────────
  Study_part:  raw=0.951 | struct=0.850 | +z=0.975 | Δ=+0.126 ⚠️
  PDL bins:    AUC=0.887 ✅

────────────────────────────────────────────────────────────
  FUSION-READINESS
────────────────────────────────────────────────────────────
  telomere_length           | simple ρ=+0.218 | parcial ρ=-0.040 (ns, p=7.272e-01) | n=77
  mtdna_cn                  | simple ρ=-0.080 | parcial ρ=+0.198 (ns, p=8.447e-02) | n=77
  PDL (sanity)              | ρ=+0.057